# Stage 2 follow-up: Derived Features
## ISM 6562 — Final Project — MediStream Telehealth
---
Closes two gaps in `02-spark-transforms.ipynb`. The brief calls out two derived features under the Stage 2 guide that the original notebook does not produce:

1. **`patient_history_score`** — "ratio of completed to scheduled appointments" per patient. Useful as a feature for no-show prediction (Q1) and to identify highly-engaged vs. flaky patients.
2. **`physician_quality_adjusted_volume`** — "appointments weighted by patient ratings" per physician. A fairer ranking signal than raw volume, since it rewards physicians who handle many patients *well* rather than just many patients.

Both share the appointments+feedback join, so they live in one notebook.

**Inputs:**
- `/medistream/curated/appointments`
- `/medistream/curated/patient_feedback`

**Outputs:**
- `/medistream/analytics/patient_history_scores` — partitioned by `status_bucket` (engagement tier)
- `/medistream/analytics/physician_quality_adjusted_volume` — single-file output (small, ~800 rows)

## Setup

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
    .appName('MediStream-Stage2d-DerivedFeatures')
    .master('spark://spark-master:7077')
    .config('spark.executor.memory', '1g')
    .config('spark.driver.memory', '1g')
    .config('spark.sql.shuffle.partitions', '8')
    .getOrCreate())

HDFS_CURATED   = 'hdfs://namenode:9000/medistream/curated'
HDFS_ANALYTICS = 'hdfs://namenode:9000/medistream/analytics'
LOCAL_CURATED   = '/home/jovyan/data/curated'
LOCAL_ANALYTICS = '/home/jovyan/data/analytics'

try:
    spark.range(1).write.mode('overwrite').parquet(f'{HDFS_ANALYTICS}/_probe')
    CURATED, ANALYTICS = HDFS_CURATED, HDFS_ANALYTICS
    print('Using HDFS')
except Exception:
    CURATED, ANALYTICS = LOCAL_CURATED, LOCAL_ANALYTICS
    print('HDFS unavailable, using local mount')

appts = spark.read.parquet(f'{CURATED}/appointments')
feedback = spark.read.parquet(f'{CURATED}/patient_feedback')
print(f'Appointments: {appts.count():,} rows')
print(f'Feedback:     {feedback.count():,} rows')

## Feature 1: `patient_history_score`

Per-patient ratio of completed appointments to total scheduled. Also breaks out cancellations and no-shows so downstream models can use them as separate features.

An `engagement_tier` partition key bucketizes patients into `loyal` (≥0.8 completion), `mixed` (0.5–0.8), `flaky` (<0.5). This makes per-tier scans cheap for model training and segment dashboards.

In [ ]:
patient_scores = (appts
    .groupBy('patient_id')
    .agg(
        F.count('*').alias('total_scheduled'),
        F.sum(F.when(F.lower(F.col('status')) == 'completed', 1).otherwise(0)).alias('completed_count'),
        F.sum(F.when(F.lower(F.col('status')) == 'no_show', 1).otherwise(0)).alias('no_show_count'),
        F.sum(F.when(F.lower(F.col('status')) == 'cancelled', 1).otherwise(0)).alias('cancelled_count'),
    )
    .withColumn('patient_history_score',
        F.round(F.col('completed_count') / F.col('total_scheduled'), 4))
    .withColumn('no_show_ratio',
        F.round(F.col('no_show_count') / F.col('total_scheduled'), 4))
    .withColumn('engagement_tier',
        F.when(F.col('patient_history_score') >= 0.8, 'loyal')
         .when(F.col('patient_history_score') >= 0.5, 'mixed')
         .otherwise('flaky'))
)

out_path = f'{ANALYTICS}/patient_history_scores'
(patient_scores.write
    .mode('overwrite')
    .partitionBy('engagement_tier')
    .parquet(out_path))

print(f'Wrote {patient_scores.count():,} patient rows to {out_path}')
print('\nEngagement tier distribution:')
patient_scores.groupBy('engagement_tier').count().orderBy(F.desc('count')).show()
print('\nSample (highest-volume patients first):')
patient_scores.orderBy(F.desc('total_scheduled')).show(10, truncate=False)

## Feature 2: `physician_quality_adjusted_volume`

For each physician, multiply their volume by their average patient rating. This produces a ranking that rewards physicians who do *high-volume, high-quality* work over those who churn through patients with low ratings.

We also retain the raw components (`raw_volume`, `feedback_count`, `avg_rating`, `avg_nps`) so analysts can decompose the score and inspect coverage gaps (physicians with too little feedback to score reliably).

In [ ]:
# Per-physician volume from appointments
phys_volume = (appts
    .groupBy('physician_id')
    .agg(F.count('*').alias('raw_volume'))
)

# Per-physician feedback stats — join feedback to appointments so we can attribute by physician_id
fb_with_phys = (feedback
    .join(appts.select('appointment_id', 'physician_id'), on='appointment_id', how='inner')
    .filter(F.col('rating').isNotNull())
)
phys_feedback = (fb_with_phys
    .groupBy('physician_id')
    .agg(
        F.count('*').alias('feedback_count'),
        F.round(F.avg('rating'), 3).alias('avg_rating'),
        F.round(F.avg('nps_score'), 2).alias('avg_nps'),
    )
)

# Find max rating across the dataset to normalize. Brief doesn't pin a scale
# (rating type is long, NPS is 0-10) so we compute it rather than assume.
max_rating = feedback.agg(F.max('rating')).first()[0]
print(f'Observed max rating value: {max_rating}')

phys_qav = (phys_volume
    .join(phys_feedback, on='physician_id', how='left')
    .withColumn('rating_multiplier',
        F.when(F.col('avg_rating').isNotNull() & (F.lit(max_rating) > 0),
               F.col('avg_rating') / F.lit(max_rating))
         .otherwise(F.lit(None).cast('double')))
    .withColumn('quality_adjusted_volume',
        F.when(F.col('rating_multiplier').isNotNull(),
               F.round(F.col('raw_volume') * F.col('rating_multiplier'), 2))
         .otherwise(F.lit(None).cast('double')))
    .withColumn('feedback_coverage_pct',
        F.round(F.coalesce(F.col('feedback_count'), F.lit(0)) / F.col('raw_volume') * 100, 2))
    .orderBy(F.desc_nulls_last('quality_adjusted_volume'))
)

out_path2 = f'{ANALYTICS}/physician_quality_adjusted_volume'
phys_qav.write.mode('overwrite').parquet(out_path2)
print(f'Wrote {phys_qav.count():,} physician rows to {out_path2}')
print('\nTop 15 by quality-adjusted volume:')
phys_qav.show(15, truncate=False)

## Verify

In [ ]:
for name in ['patient_history_scores', 'physician_quality_adjusted_volume']:
    df = spark.read.parquet(f'{ANALYTICS}/{name}')
    print(f'  {name:<40s}  {df.count():>8,} rows, {len(df.columns):>3} cols  OK')